# Classification multimodale image + texte

Ce notebook combine les modèles unimodaux image et texte précédemment entraînés. L'objectif est d'évaluer si la fusion des deux modalités améliore la classification multi-label par rapport aux approches image seule et texte seule.

Le notebook multimodal ne réentraîne pas les modèles image et texte depuis zéro. Il les recharge depuis le registre des modèles, prépare les entrées synchronisées, puis compare plusieurs stratégies de fusion.


## 1. Objectif du notebook

- Combiner deux modalités : image et texte.
- Réutiliser le meilleur modèle image entraîné dans `image_classifier.ipynb`.
- Réutiliser le meilleur modèle texte entraîné dans `text_classiffier.ipynb`.
- Comparer les performances unimodales et multimodales.
- Sélectionner et sauvegarder le meilleur modèle multimodal.


## 2. Chargement des librairies et des chemins

Cette partie doit importer les bibliothèques nécessaires et définir les chemins du projet :

- `train.csv`,
- dossier contenant les images,
- `model/registry/image_model.keras`,
- `model/registry/text_model.keras`,
- `model/registry/text_tokenizer.pkl` ou vectorizer texte,
- `model/registry/multilabel_binarizer.pkl`,
- chemin de sauvegarde du modèle multimodal final.


## 3. Chargement du dataset commun

Cette partie doit charger le fichier annoté `train.csv` et conserver les colonnes utiles :

- `ImageID`,
- `Caption`,
- `Labels`.

Les champs doivent être nettoyés de façon minimale : espaces inutiles, captions vides, labels vides et vérification de l'existence des images associées.


## 4. Split commun train / validation / test

Le split doit rester identique entre les notebooks image, texte et multimodal :

- 70% train,
- 15% validation,
- 15% test.

Le même `random_state` doit être utilisé afin de garder les instances synchronisées entre les images, les captions et les labels.


## 5. Encodage des labels multi-label

Les labels constituent la cible supervisée commune aux trois approches :

- image seule,
- texte seul,
- image + texte.

Cette partie doit transformer les labels en vecteurs binaires multi-label à l'aide d'un `MultiLabelBinarizer`, chargé depuis le registre si disponible, ou reconstruit avec les mêmes classes que les notebooks unimodaux.


## 6. Chargement des modèles unimodaux sauvegardés

Cette partie doit charger les artefacts produits par les notebooks unimodaux :

- modèle image sauvegardé,
- modèle texte sauvegardé,
- tokenizer ou vectorizer texte,
- encodeur des labels.

Il faut également vérifier les formats d'entrée attendus par chaque modèle : taille des images, longueur maximale des séquences texte et nombre de labels en sortie.


## 7. Préparation des entrées multimodales

Cette partie doit préparer les entrées synchronisées pour chaque exemple :

- image prétraitée : lecture, redimensionnement et normalisation,
- caption prétraitée : tokenisation et padding,
- labels multi-label encodés.

Le dataset final doit permettre d'alimenter un modèle avec deux entrées : `image` et `text_tokens`.


## 8. Évaluation des modèles unimodaux

Avant de tester la fusion, il faut mesurer les performances de référence :

- prédictions du modèle image seul,
- prédictions du modèle texte seul.

Les métriques utilisées doivent être adaptées à la classification multi-label :

- F1 micro,
- F1 macro,
- Average Precision,
- AUC.


## 9. Fusion tardive des prédictions

La fusion tardive combine directement les scores de sortie des modèles unimodaux.

Exemples de stratégies :

- moyenne simple des scores image et texte,
- moyenne pondérée si une modalité est plus performante que l'autre.

Cette approche sert de baseline multimodale simple et facilement interprétable.


## 10. Fusion par caractéristiques

La fusion par caractéristiques utilise les représentations internes des modèles image et texte.

Principe :

- retirer la dernière couche de classification du modèle image,
- retirer la dernière couche de classification du modèle texte,
- extraire les features image et texte,
- concaténer les deux représentations,
- ajouter une tête Dense multi-label avec sortie `sigmoid`.

Cette approche permet au modèle multimodal d'apprendre une représentation commune image + texte.


## 11. Entraînement du modèle multimodal

Cette partie entraîne la tête de fusion multimodale.

Deux stratégies peuvent être comparées :

- branches image et texte gelées, seule la tête de fusion est entraînée,
- fine-tuning léger d'une partie des branches si les performances le justifient.

La loss utilisée doit être `binary_crossentropy`, car une image peut appartenir à plusieurs labels.


## 12. Comparaison finale

Cette partie doit comparer toutes les approches :

- `image_only`,
- `text_only`,
- `late_fusion`,
- `feature_fusion`.

La comparaison doit être présentée sous forme de tableau avec les métriques multi-label retenues. Le choix du modèle final doit être justifié à partir des résultats.


## 13. Analyse qualitative des résultats

Cette partie doit afficher quelques exemples de test avec :

- l'image,
- la caption,
- les vrais labels,
- les prédictions image,
- les prédictions texte,
- les prédictions multimodales.

L'objectif est d'identifier les cas où l'image aide, les cas où le texte aide, les cas où la fusion améliore la prédiction et les cas où elle échoue.


## 14. Sauvegarde du modèle multimodal

Le modèle multimodal retenu doit être sauvegardé dans `model/registry/`.

Les métadonnées associées peuvent inclure :

- type de fusion utilisée,
- métriques obtenues,
- date d'entraînement,
- modèles unimodaux utilisés,
- paramètres principaux du preprocessing.


## 15. Conclusion

Cette conclusion doit résumer :

- l'intérêt de la fusion multimodale,
- la comparaison entre image seule, texte seul et image + texte,
- le modèle final retenu,
- les limites observées,
- les pistes d'amélioration.
